### Naive Bayes — Kaggle Playground Series S6E4: Predicting Irrigation Need
GSBS545 In-Class Activity — April 28, 2026

In [ ]:
# import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, balanced_accuracy_score, classification_report)

### Import, inspect, and split data

In [ ]:
# load kaggle train data
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
print(train.shape)
print()
print(train["Irrigation_Need"].value_counts())
print()
print("missing values:", train.isna().sum().sum())
print()
print(train.head())

In [ ]:
# split features and target
X = train.drop(["id", "Irrigation_Need"], axis=1)
y = train["Irrigation_Need"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

### Identify column types and define preprocessing

In [ ]:
# identify numeric and categorical columns
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("numeric columns:", numeric_features)
print("categorical columns:", categorical_features)

In [ ]:
# helper: convert sparse matrices to dense arrays (required for GaussianNB)
def to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else x

# define preprocessor
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

### Build and evaluate Gaussian Naive Bayes

In [ ]:
# define model pipeline
nb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("to_dense", FunctionTransformer(to_dense)),
    ("model", GaussianNB())
])

# cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(nb_pipeline, X_train, y_train, cv=cv,
                        scoring=["accuracy", "balanced_accuracy"])

print("CV accuracy:", round(scores["test_accuracy"].mean(), 4))
print("CV balanced accuracy:", round(scores["test_balanced_accuracy"].mean(), 4))

In [ ]:
# fit on full training set and evaluate on held-out test set
nb_pipeline.fit(X_train, y_train)
preds_baseline = nb_pipeline.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, preds_baseline), 4))
print("Test balanced accuracy:", round(balanced_accuracy_score(y_test, preds_baseline), 4))

### Generate predicted probabilities and baseline predictions

In [ ]:
# predicted probabilities
probs = nb_pipeline.predict_proba(X_test)
classes = nb_pipeline.classes_

print("Classes:", classes)
print("Probability matrix shape:", probs.shape)
print()

# baseline predictions = default rule (choose class with highest probability)
preds_baseline = probs.argmax(axis=1)
# map back to class labels
preds_baseline_labels = classes[preds_baseline]

print("Baseline classification report:")
print(classification_report(y_test, preds_baseline_labels))
print("Baseline balanced accuracy:", round(balanced_accuracy_score(y_test, preds_baseline_labels), 4))

### Threshold tuning for the "High" class

I chose the **High** class because it is the rarest (only ~3.3% of samples) and is the most likely to be underrepresented in baseline predictions. My evaluation metric is **balanced accuracy**, which accounts for class imbalance by averaging per-class recall.

In [ ]:
# find the index of the "High" class in the probability matrix
target_class = "High"
target_idx = list(classes).index(target_class)
print(f"Target class: {target_class} (index {target_idx})")

# test a range of thresholds for the High class
thresholds = np.linspace(0.05, 0.90, 35)
rows = []

for t in thresholds:
    preds_threshold = preds_baseline.copy()
    mask = probs[:, target_idx] >= t
    preds_threshold[mask] = target_idx
    preds_threshold_labels = classes[preds_threshold]
    
    rows.append({
        "threshold": round(t, 4),
        "balanced_accuracy": balanced_accuracy_score(y_test, preds_threshold_labels),
        "recall_High": recall_score(y_test, preds_threshold_labels, labels=[target_class], average=None)[0],
        "f1_High": f1_score(y_test, preds_threshold_labels, labels=[target_class], average=None)[0]
    })

threshold_df = pd.DataFrame(rows)
print(threshold_df.sort_values("balanced_accuracy", ascending=False).head(10))

In [ ]:
# apply the best threshold
best_row = threshold_df.loc[threshold_df["balanced_accuracy"].idxmax()]
best_threshold = best_row["threshold"]
print(f"Best threshold for High class: {best_threshold}")
print()

# generate threshold-adjusted predictions
preds_threshold = preds_baseline.copy()
mask = probs[:, target_idx] >= best_threshold
preds_threshold[mask] = target_idx
preds_threshold_labels = classes[preds_threshold]

print("Threshold-adjusted classification report:")
print(classification_report(y_test, preds_threshold_labels))
print("Threshold-adjusted balanced accuracy:", round(balanced_accuracy_score(y_test, preds_threshold_labels), 4))

### Comparison: baseline vs threshold-adjusted predictions

In [ ]:
# side-by-side comparison
print("=== BASELINE (default threshold) ===")
print(classification_report(y_test, classes[preds_baseline]))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, classes[preds_baseline]), 4))
print()
print(f"=== THRESHOLD-ADJUSTED (High >= {best_threshold}) ===")
print(classification_report(y_test, preds_threshold_labels))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, preds_threshold_labels), 4))

### Discussion

**Class selected:** High — the rarest class at ~3.3% of the dataset (21,009 of 630,000 samples). This is the class most likely to benefit from threshold tuning since rare classes are often underrepresented in default predictions.

**Threshold chosen:** 0.45. This is slightly below the default argmax decision boundary, meaning the model assigns "High" when it estimates at least a 45% probability rather than requiring "High" to be the single most likely class.

**How the metric changed:** The baseline model already achieved 0.7794 balanced accuracy with 78% recall on the High class — surprisingly strong for a rare class. Lowering the threshold to 0.45 only improved balanced accuracy marginally to 0.7797. The High class recall stayed at 78%, precision dropped slightly from 0.53 to 0.50, and Medium recall dropped from 0.77 to 0.76. The threshold had minimal impact because Naive Bayes was already reasonably confident in its High predictions — most true High samples already had High as their argmax class.

**Tradeoff observed:** This is a case where the precision-recall tradeoff is modest. Lowering the threshold slightly increased the number of High predictions (more false positives, lower precision) without meaningfully improving recall. The tradeoff was not dramatic here because the baseline model was not systematically under-predicting High — unlike what we might expect with a 3.3% base rate.

**Naive Bayes vs existing models:** Gaussian Naive Bayes achieved ~0.78 balanced accuracy. This is substantially lower than tree-based ensemble models (LightGBM, XGBoost, CatBoost), which achieve CV balanced accuracy above 0.97 on this dataset. Naive Bayes underperforms here because it assumes feature independence and Gaussian-distributed features within each class — neither assumption holds well. The categorical features are one-hot encoded into binary columns (clearly not Gaussian), and features like Soil_Moisture, Humidity, and Rainfall interact in ways that an independence assumption misses. Still, Naive Bayes is useful as a fast, interpretable baseline and for demonstrating probability calibration and threshold tuning concepts.